# 09 — Real Anonymized Speech: Content Evaluation

## Objective

Run the existing content-preservation framework on **real original/anonymized speech pairs** generated with an official VoicePrivacy Challenge 2026 baseline.

```
Original audio + anonymized audio
              ↓
          Faster-Whisper
              ↓
     original / anonymized transcripts
              ↓
    ┌─────────┼──────────┐
    ↓         ↓          ↓
Similarity   NLI   Semantic alignment → TPI
```

This notebook orchestrates the existing metrics. It does not redefine them.

## Experimental setup

- VoicePrivacy Challenge 2026 Track 1, LibriSpeech dev subset.
- Official B2 McAdams anonymization.
- Initial size: 100 paired utterances.
- The 100-pair manifest is built directly in this notebook from the VPC subset files.
- Same Faster-Whisper configuration for original and anonymized audio.
- Content metrics remain separate; no arbitrary combined loss yet.

The notebook therefore starts from the already-generated B2 WAV files and handles the pairing, ASR, and content evaluation end-to-end.


In [1]:

import sys

import numpy as np
import pandas as pd

# Locate the project root from the current notebook directory.
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / "src" / "paths.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing src/paths.py."
    )

# Ensure local project modules take precedence over site-packages.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load centralized project paths directly to avoid conflicts with
# third-party packages named "src".
import importlib.util

paths_spec = importlib.util.spec_from_file_location(
    "project_paths",
    PROJECT_ROOT / "src" / "paths.py",
)

if paths_spec is None or paths_spec.loader is None:
    raise ImportError("Could not load src/paths.py.")

paths = importlib.util.module_from_spec(paths_spec)
paths_spec.loader.exec_module(paths)

# VPC 2026 experiment paths.
DATA_DIR = paths.DATA_DIR / "vpc2026"
RESULTS_DIR = paths.DATA_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SUBSET_DIR = DATA_DIR / "subset_100"
ORIGINAL_DATA_DIR = DATA_DIR / "data" / "libri_dev"
ANON_WAV_DIR = DATA_DIR / "subset_100_b2" / "wav"

PAIR_MANIFEST = DATA_DIR / "pairs_100.csv"
TRANSCRIPT_OUTPUT = RESULTS_DIR / "vpc2026_b2_100_transcripts.csv"
METRICS_OUTPUT = RESULTS_DIR / "vpc2026_b2_100_content_metrics.csv"

print("Project root:", PROJECT_ROOT)
print("VPC data:", DATA_DIR)
print("Subset:", SUBSET_DIR)
print("Original data:", ORIGINAL_DATA_DIR)
print("B2 anonymized WAVs:", ANON_WAV_DIR)
print("Results:", RESULTS_DIR)


Project root: /Users/juni/Documents/interview-anonymization
VPC data: /Users/juni/Documents/interview-anonymization/data/vpc2026
Subset: /Users/juni/Documents/interview-anonymization/data/vpc2026/subset_100
Original data: /Users/juni/Documents/interview-anonymization/data/vpc2026/data/libri_dev
B2 anonymized WAVs: /Users/juni/Documents/interview-anonymization/data/vpc2026/subset_100_b2/wav
Results: /Users/juni/Documents/interview-anonymization/data/results


In [2]:
# Build the 100 original/anonymized pairs directly from the VPC subset.
# No separate make_pairs_100.py script is required.

WAV_SCP = SUBSET_DIR / "wav.scp"
UTT2SPK = SUBSET_DIR / "utt2spk"

for required_file in [WAV_SCP, UTT2SPK, ANON_WAV_DIR]:
    if not required_file.exists():
        raise FileNotFoundError(f"Missing required VPC output: {required_file}")

# Speaker mapping
utt2spk = {}
with open(UTT2SPK) as f:
    for line in f:
        utt, spk = line.strip().split()
        utt2spk[utt] = spk

# Pair original and B2-anonymized audio using the utterance ID.
pair_rows = []

with open(WAV_SCP) as f:
    for line in f:
        utt, rel_path = line.strip().split(maxsplit=1)

        # wav.scp paths are relative to data/vpc2026.
        original_path = DATA_DIR / rel_path
        anonymized_path = ANON_WAV_DIR / f"{utt}.wav"

        if not original_path.exists():
            raise FileNotFoundError(f"Missing original audio: {original_path}")
        if not anonymized_path.exists():
            raise FileNotFoundError(f"Missing anonymized audio: {anonymized_path}")
        if utt not in utt2spk:
            raise ValueError(f"Speaker ID missing for utterance: {utt}")

        pair_rows.append({
            "pair_id": utt,
            "original_path": str(original_path),
            "anonymized_path": str(anonymized_path),
            "speaker_id": utt2spk[utt],
        })

pairs_df = pd.DataFrame(pair_rows)

if len(pairs_df) != 100:
    raise ValueError(f"Expected exactly 100 pairs, found {len(pairs_df)}")

pairs_df.to_csv(PAIR_MANIFEST, index=False)

print(f"Created pair manifest: {PAIR_MANIFEST}")
print(f"Pairs loaded: {len(pairs_df)}")
print(f"Unique speakers: {pairs_df['speaker_id'].nunique()}")

display(pairs_df.head())


Created pair manifest: /Users/juni/Documents/interview-anonymization/data/vpc2026/pairs_100.csv
Pairs loaded: 100
Unique speakers: 40


,pair_id,original_path,anonymized_path,speaker_id
0,1462-170142-0001,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,1462
1,251-137823-0004,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,251
2,174-168635-0014,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,174
3,3170-137482-0040,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,3170
4,7850-73752-0004,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,7850


### Pairing checkpoint

At this point the notebook has verified the experimental input:

**100 original WAVs ↔ 100 B2-anonymized WAVs.**

The generated `pairs_100.csv` is the reproducible manifest used by all subsequent cells.


In [3]:
from faster_whisper import WhisperModel
import importlib.util

# Load project utilities without importing the ambiguous "src" package.
utils_spec = importlib.util.spec_from_file_location(
    "project_utils",
    PROJECT_ROOT / "src" / "utils.py",
)

if utils_spec is None or utils_spec.loader is None:
    raise ImportError("Could not load src/utils.py.")

utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(utils)

# Load the shared ASR implementation.
asr_module = utils.load_module(
    "project_asr",
    PROJECT_ROOT / "src" / "asr.py",
)

# ASR configuration.
ASR_MODEL_NAME = "base"
ASR_DEVICE = "cpu"
ASR_COMPUTE_TYPE = "int8"
ASR_LANGUAGE = "en"

# Load the model once.
asr_model = WhisperModel(
    ASR_MODEL_NAME,
    device=ASR_DEVICE,
    compute_type=ASR_COMPUTE_TYPE,
)

# Reuse the functions implemented in src/asr.py.
transcribe_audio = asr_module.transcribe_audio
transcribe_pairs = asr_module.transcribe_pairs

print(f"ASR model: {ASR_MODEL_NAME}")
print(f"Device: {ASR_DEVICE}")
print(f"Compute type: {ASR_COMPUTE_TYPE}")
print(f"Language: {ASR_LANGUAGE}")
print("ASR functions loaded from src/asr.py.")


/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ASR model: base
Device: cpu
Compute type: int8
Language: en
ASR functions loaded from src/asr.py.


In [4]:

required_columns = [
    "pair_id",
    "original_path",
    "anonymized_path",
    "original_transcript",
    "anonymized_transcript",
]

if TRANSCRIPT_OUTPUT.exists():
    transcript_df = pd.read_csv(TRANSCRIPT_OUTPUT)

    missing_columns = [
        column
        for column in required_columns
        if column not in transcript_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Cached transcripts are missing columns: {missing_columns}"
        )

    if transcript_df["pair_id"].duplicated().any():
        raise ValueError("Cached transcripts contain duplicate pair IDs.")

    expected_ids = set(pairs_df["pair_id"])
    cached_ids = set(transcript_df["pair_id"])

    if cached_ids != expected_ids:
        raise ValueError(
            "Cached transcript IDs do not match the current audio pairs. "
            "Check the cache before rerunning transcription."
        )

    print(f"Loaded cached transcripts: {len(transcript_df)} pairs")

else:
    transcript_df = transcribe_pairs(
        pairs_df,
        model=asr_model,
        language=ASR_LANGUAGE,
    )

    if len(transcript_df) != len(pairs_df):
        raise ValueError(
            f"Expected {len(pairs_df)} transcript pairs, "
            f"found {len(transcript_df)}."
        )

    transcript_df.to_csv(TRANSCRIPT_OUTPUT, index=False)

    print(f"Transcribed and saved {len(transcript_df)} pairs")

if transcript_df[required_columns].isna().any().any():
    raise ValueError("Transcript data contains missing required values.")

display(
    transcript_df[
        ["pair_id", "original_transcript", "anonymized_transcript"]
    ].head(10)
)


Loaded cached transcripts: 100 pairs


,pair_id,original_transcript,anonymized_transcript
0,1462-170142-0001,Emerging at Houston at half past three o'clock...,"Emerging at Euston, I have passed three o'cloc..."
1,251-137823-0004,Electronic equipment cascaded from the wall sh...,The electronic equipment cascaded from the wal...
2,174-168635-0014,Cousette was no longer in rags. She was in mou...,Cusette was no longer in rags. She was in mour...
3,3170-137482-0040,"But, although, believing fully in my oracles, ...","But, although, believing fully in my oracles, ..."
4,7850-73752-0004,"Ferdinand felt his freedom, as well as his hap...","Ferdinand fell to his freedom, as well as his ..."
5,1272-135031-0004,The king is flooded disgrace and your friends ...,"The king is planning to disgrace, and your fri..."
6,1993-147964-0000,"They sought about the house most of the day, a...","They sought about the house most of the day, a..."
7,6241-61943-0000,"On the second of the month, at 2 in the mornin...","On a second and a month, at 2 in the morning, ..."
8,3752-4943-0012,"Let him out, cried north again, stamping his f...","I don't know how to climb Earth again, stabbin..."
9,2035-147960-0015,I don't winked at me.,I don't want it to mean.


## 2. Load the shared semantic-alignment + TPI implementation

The semantic-alignment and TPI implementations are maintained in `src/content/`.

This notebook imports the shared evaluation pipeline rather than duplicating
the implementation. It loads the required language model once and reuses it
across all original/anonymized transcript pairs.

In [5]:
import sys

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Make the src/ directory importable as a Python module location.
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Check that the required source files exist.
CONTENT_DIR = SRC_DIR / "content"

required_modules = [
    CONTENT_DIR / "semantic_alignment.py",
    CONTENT_DIR / "tpi.py",
    CONTENT_DIR / "evaluation.py",
]

for module_path in required_modules:
    if not module_path.is_file():
        raise FileNotFoundError(
            f"Required content module not found: {module_path}"
        )

# Import the shared implementation as a proper Python package.
from content.evaluation import evaluate_content_pair

# Experimental configuration.
MODEL_NAME = "bert-base-uncased"
SIMILARITY_THRESHOLD = 0.80
TPI_STRATEGY = "N-TA"
N = 6

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load the model once.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
).to(DEVICE)

model.eval()

print("Loaded content-evaluation implementation from src/content/.")
print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Similarity threshold:", SIMILARITY_THRESHOLD)
print("TPI strategy:", TPI_STRATEGY, "| N:", N)


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 202/202 [00:00<00:00, 5425.56it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded content-evaluation implementation from src/content/.
Model: bert-base-uncased
Device: cpu
Similarity threshold: 0.8
TPI strategy: N-TA | N: 6


## 3. Semantic similarity

The selected model from notebook 05c is `paraphrase-mpnet-base-v2`. Use the same model for both transcripts.

In [6]:
from sentence_transformers import SentenceTransformer

SIM_MODEL_NAME = "sentence-transformers/paraphrase-mpnet-base-v2"

# Validate the input transcripts.
transcript_columns = [
    "original_transcript",
    "anonymized_transcript",
]

for column in transcript_columns:
    if column not in transcript_df.columns:
        raise ValueError(f"Missing required column: {column}")

    if transcript_df[column].isna().any():
        raise ValueError(f"Missing transcripts in column: {column}")

    if transcript_df[column].astype(str).str.strip().eq("").any():
        raise ValueError(f"Empty transcripts in column: {column}")

# Load the shared similarity implementation from src/content/embeddings.py.
embeddings_module = utils.load_module(
    "content_embeddings",
    PROJECT_ROOT / "src" / "content" / "embeddings.py",
)

# Load the model once.
sim_model = SentenceTransformer(SIM_MODEL_NAME)

# Pair each original transcript with its anonymized counterpart.
pairs_by_category = {
    "original_vs_anonymized": list(
        zip(
            transcript_df["original_transcript"].tolist(),
            transcript_df["anonymized_transcript"].tolist(),
        )
    )
}

# Reuse the shared implementation.
similarity_results = embeddings_module.evaluate_text_pairs(
    model=sim_model,
    pairs_by_category=pairs_by_category,
)

# Validate the output before adding it to the main dataframe.
if len(similarity_results) != len(transcript_df):
    raise ValueError(
        "Similarity result count does not match transcript count: "
        f"{len(similarity_results)} vs {len(transcript_df)}"
    )

scores = similarity_results["cosine"].to_numpy()

if not np.isfinite(scores).all():
    raise ValueError("Semantic similarity contains non-finite values.")

transcript_df["semantic_similarity"] = scores

print(
    f"Semantic similarity computed for {len(transcript_df)} pairs "
    f"using {SIM_MODEL_NAME}."
)


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 5498.79it/s]


Semantic similarity computed for 100 pairs using sentence-transformers/paraphrase-mpnet-base-v2.


## 4. NLI

NLI is used as a complementary content-preservation signal.

The exact model/configuration selected in notebook 06b is loaded below:
`MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli`.

For each original/anonymized transcript pair, we keep the three NLI probabilities
(entailment, neutral, contradiction) and the highest-probability predicted relation.

Do **not** combine NLI, TPI, and semantic similarity into one arbitrary loss at this stage.


In [7]:

# Load the shared NLI implementation from src/content/nli.py.
nli_module = utils.load_module(
    "content_nli",
    PROJECT_ROOT / "src" / "content" / "nli.py",
)

NLI_MODEL_NAME = (
    "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
)
NLI_BATCH_SIZE = 8

nli_tokenizer, nli_model, NLI_DEVICE, NLI_LABEL_MAPPING = (
    nli_module.load_nli_model(NLI_MODEL_NAME)
)

print(f"NLI model loaded: {NLI_MODEL_NAME}")
print(f"Device: {NLI_DEVICE}")
print(f"Label mapping: {NLI_LABEL_MAPPING}")


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 394/394 [00:00<00:00, 5511.57it/s]


NLI model loaded: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Device: mps
Label mapping: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


In [8]:

nli_pairs = list(
    zip(
        transcript_df["original_transcript"].tolist(),
        transcript_df["anonymized_transcript"].tolist(),
    )
)

nli_results = nli_module.predict_nli(
    tokenizer=nli_tokenizer,
    model=nli_model,
    pairs=nli_pairs,
    device=NLI_DEVICE,
    label_mapping=NLI_LABEL_MAPPING,
    batch_size=NLI_BATCH_SIZE,
)

if len(nli_results) != len(transcript_df):
    raise ValueError(
        "NLI result count does not match transcript count: "
        f"{len(nli_results)} vs {len(transcript_df)}"
    )

nli_results.insert(
    0,
    "pair_id",
    transcript_df["pair_id"].to_numpy(),
)

nli_results.insert(
    1,
    "sentence_1",
    transcript_df["original_transcript"].to_numpy(),
)

nli_results.insert(
    2,
    "sentence_2",
    transcript_df["anonymized_transcript"].to_numpy(),
)

nli_results["predicted"] = nli_results[
    ["entailment", "neutral", "contradiction"]
].idxmax(axis=1)

print(f"NLI predictions computed for {len(nli_results)} pairs.")
display(nli_results.head())


NLI predictions computed for 100 pairs.


,pair_id,sentence_1,sentence_2,entailment,neutral,contradiction,predicted
0,1462-170142-0001,Emerging at Houston at half past three o'clock...,"Emerging at Euston, I have passed three o'cloc...",0.205444,0.021011,0.773438,contradiction
1,251-137823-0004,Electronic equipment cascaded from the wall sh...,The electronic equipment cascaded from the wal...,0.998047,0.001343,0.000160,entailment
2,174-168635-0014,Cousette was no longer in rags. She was in mou...,Cusette was no longer in rags. She was in mour...,0.979004,0.012764,0.007988,entailment
3,3170-137482-0040,"But, although, believing fully in my oracles, ...","But, although, believing fully in my oracles, ...",0.548828,0.422119,0.028793,entailment
4,7850-73752-0004,"Ferdinand felt his freedom, as well as his hap...","Ferdinand fell to his freedom, as well as his ...",0.006142,0.986328,0.006985,neutral


## 5. Semantic alignment + TPI on real pairs

In [9]:
metric_rows = []

for _, row in transcript_df.iterrows():
    result = evaluate_content_pair(
        row.original_transcript,
        row.anonymized_transcript,
        tokenizer=tokenizer,
        model=model,
        device=DEVICE,
        similarity_threshold=SIMILARITY_THRESHOLD,
        tpi_strategy=TPI_STRATEGY,
        N=N,
    )

    metric_rows.append({
        "pair_id": row.pair_id,
        "TPI": result["TPI"],
        "L_TPI": result["L_TPI"],
        "TIC_original": result["tpi"]["TIC_original"],
        "information_lost": result["tpi"]["information_lost"],
        "n_affected_terms": len(result["alignment"]["affected_original_indices"]),
        "affected_terms": ", ".join(result["alignment"]["affected_original_terms"]),
    })

metrics_df = pd.DataFrame(metric_rows)

# Add semantic similarity.
metrics_df = metrics_df.merge(
    transcript_df[["pair_id", "semantic_similarity"]],
    on="pair_id",
    how="left",
)

# Add the NLI probabilities and predicted relation.
metrics_df = metrics_df.merge(
    nli_results[
        [
            "pair_id",
            "entailment",
            "neutral",
            "contradiction",
            "predicted",
        ]
    ],
    on="pair_id",
    how="left",
)

metrics_df.to_csv(METRICS_OUTPUT, index=False)

display(metrics_df.head(10))


,pair_id,TPI,L_TPI,TIC_original,information_lost,n_affected_terms,affected_terms,semantic_similarity,entailment,neutral,contradiction,predicted
0,1462-170142-0001,0.697676,0.302324,102.866319,31.098998,6,"Houston, at, half, past, it, once",0.853957,0.205444,0.021011,0.773438,contradiction
1,251-137823-0004,0.850371,0.149629,104.484821,15.634005,5,"Electronic, and, a, duty, chain",0.982177,0.998047,0.001343,0.000160,entailment
2,174-168635-0014,0.486268,0.513732,31.060297,15.956676,1,Cousette,0.819615,0.979004,0.012764,0.007988,entailment
3,3170-137482-0040,1.000000,0.000000,135.504357,0.000000,0,,0.997856,0.548828,0.422119,0.028793,entailment
4,7850-73752-0004,0.809683,0.190317,29.274006,5.571328,1,felt,0.891451,0.006142,0.986328,0.006985,neutral
5,1272-135031-0004,0.433205,0.566795,55.237204,31.308188,2,"flooded, disgrace",0.855182,0.014870,0.075928,0.909180,contradiction
6,1993-147964-0000,1.000000,0.000000,102.864029,0.000000,0,,0.971320,0.982910,0.016037,0.001469,entailment
7,6241-61943-0000,0.419391,0.580609,58.796735,34.137902,13,"On, the, second, of, the, month, precious, car...",0.660140,0.004200,0.010719,0.984375,contradiction
8,3752-4943-0012,0.000000,1.000000,51.218174,51.218174,9,"Let, him, out, cried, north, again, stamping, ...",0.429771,0.000220,0.000682,0.999023,contradiction
9,2035-147960-0015,0.000000,1.000000,37.466213,37.466213,5,"I, don't, winked, at, me",0.541139,0.003374,0.996094,0.000578,neutral


## 6. Descriptive statistics and cross-metric diagnostics

Keep semantic similarity, TPI, and NLI as separate content-preservation signals.

The NLI probabilities are continuous scores, while `predicted` is a categorical
summary of the highest-probability NLI relation. We therefore report:

1. descriptive statistics for semantic similarity, TPI, and the three NLI probabilities;
2. the distribution of predicted NLI relations;
3. pairwise correlations between the continuous metrics;
4. disagreement cases for qualitative inspection.

Do **not** combine the metrics into one arbitrary loss at this stage.


In [10]:
continuous_metrics = [
    "semantic_similarity",
    "TPI",
    "L_TPI",
    "n_affected_terms",
    "entailment",
    "neutral",
    "contradiction",
]

print("Number of pairs:", len(metrics_df))

display(
    metrics_df[continuous_metrics].describe()
)

print("Predicted NLI relation proportions:")
display(
    metrics_df["predicted"]
    .value_counts(normalize=True, dropna=False)
    .rename("proportion")
    .to_frame()
)


Number of pairs: 100


,semantic_similarity,TPI,L_TPI,n_affected_terms,entailment,neutral,contradiction
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,0.831410,0.702596,0.297404,3.700000,0.554873,0.213113,0.231925
std,0.226919,0.356280,0.356280,6.137318,0.455107,0.339295,0.380119
min,0.121958,0.000000,0.000000,0.000000,0.000168,0.000660,0.000081
25%,0.755719,0.477946,0.000000,0.000000,0.008175,0.007483,0.000798
50%,0.938614,0.858516,0.141484,1.000000,0.829834,0.028023,0.005417
75%,1.000000,1.000000,0.522054,5.250000,0.987549,0.276917,0.362549
max,1.000000,1.000000,1.000000,40.000000,0.999023,0.996094,0.999023


Predicted NLI relation proportions:


,proportion
predicted,
entailment,0.58
contradiction,0.23
neutral,0.19


In [11]:
print("Correlation matrix:")

display(
    metrics_df[
        [
            "semantic_similarity",
            "TPI",
            "entailment",
            "neutral",
            "contradiction",
        ]
    ].corr()
)


Correlation matrix:


,semantic_similarity,TPI,entailment,neutral,contradiction
semantic_similarity,1.000000,0.882071,0.621592,-0.359721,-0.423145
TPI,0.882071,1.000000,0.611844,-0.353724,-0.416825
entailment,0.621592,0.611844,1.000000,-0.575630,-0.683648
neutral,-0.359721,-0.353724,-0.575630,1.000000,-0.203247
contradiction,-0.423145,-0.416825,-0.683648,-0.203247,1.000000


In [12]:
print("TPI quartiles:")
display(
    metrics_df["TPI"].quantile([0, 0.25, 0.50, 0.75, 1.00]).rename("TPI")
)

print("Semantic similarity quartiles:")
display(
    metrics_df["semantic_similarity"]
    .quantile([0, 0.25, 0.50, 0.75, 1.00])
    .rename("semantic_similarity")
)

print("NLI entailment quartiles:")
display(
    metrics_df["entailment"]
    .quantile([0, 0.25, 0.50, 0.75, 1.00])
    .rename("entailment")
)

print("NLI contradiction quartiles:")
display(
    metrics_df["contradiction"]
    .quantile([0, 0.25, 0.50, 0.75, 1.00])
    .rename("contradiction")
)


TPI quartiles:


0.00    0.000000
0.25    0.477946
0.50    0.858516
0.75    1.000000
1.00    1.000000
Name: TPI, dtype: float64

Semantic similarity quartiles:


0.00    0.121958
0.25    0.755719
0.50    0.938614
0.75    1.000000
1.00    1.000000
Name: semantic_similarity, dtype: float64

NLI entailment quartiles:


0.00    0.000168
0.25    0.008175
0.50    0.829834
0.75    0.987549
1.00    0.999023
Name: entailment, dtype: float64

NLI contradiction quartiles:


0.00    0.000081
0.25    0.000798
0.50    0.005417
0.75    0.362549
1.00    0.999023
Name: contradiction, dtype: float64

In [13]:
# Cases where semantic similarity is high but TPI is low.
# These are useful for checking whether the two metrics disagree on content preservation.

low_tpi_high_sim = metrics_df[
    (metrics_df["TPI"] < 0.50) &
    (metrics_df["semantic_similarity"] > 0.90)
].copy()

print(
    "Potential high-similarity / low-TPI cases:",
    len(low_tpi_high_sim),
)

display(
    transcript_df[
        ["pair_id", "original_transcript", "anonymized_transcript"]
    ].merge(
        low_tpi_high_sim[
            [
                "pair_id",
                "TPI",
                "semantic_similarity",
                "entailment",
                "neutral",
                "contradiction",
                "predicted",
            ]
        ],
        on="pair_id",
        how="inner",
    )[
        [
            "pair_id",
            "original_transcript",
            "anonymized_transcript",
            "semantic_similarity",
            "TPI",
            "entailment",
            "neutral",
            "contradiction",
            "predicted",
        ]
    ].head(10)
)


Potential high-similarity / low-TPI cases: 2


,pair_id,original_transcript,anonymized_transcript,semantic_similarity,TPI,entailment,neutral,contradiction,predicted
0,2078-142845-0024,1724,1724.,0.963714,0.000000,0.995117,0.004326,0.000684,entailment
1,1673-143397-0008,"At these blasts from a sound, the pillars of t...","At these blocks from the sounds, the pillars o...",0.904375,0.452982,0.898926,0.047180,0.053467,entailment


In [14]:
# Potential NLI / similarity disagreement:
# high semantic similarity but low entailment probability.

high_sim_low_entailment = metrics_df[
    (metrics_df["semantic_similarity"] >
     0.90) &
    (metrics_df["entailment"] <
     0.50)
].copy()

print(
    "Potential high-similarity / low-entailment cases:",
    len(high_sim_low_entailment),
)

display(
    transcript_df[
        ["pair_id", "original_transcript", "anonymized_transcript"]
    ].merge(
        high_sim_low_entailment[
            [
                "pair_id",
                "semantic_similarity",
                "TPI",
                "entailment",
                "neutral",
                "contradiction",
                "predicted",
            ]
        ],
        on="pair_id",
        how="inner",
    )[
        [
            "pair_id",
            "original_transcript",
            "anonymized_transcript",
            "semantic_similarity",
            "TPI",
            "entailment",
            "neutral",
            "contradiction",
            "predicted",
        ]
    ].head(10)
)


Potential high-similarity / low-entailment cases: 7


,pair_id,original_transcript,anonymized_transcript,semantic_similarity,TPI,entailment,neutral,contradiction,predicted
0,6319-57405-0005,"The day is coming,"" said Pometheus, when Jupit...","The day is coming, sepomethious, when Jupiter ...",0.902973,0.776897,0.043060,0.943848,0.013466,neutral
1,251-137823-0000,"I'll be glad to try, sir,"" he replied.","I'll be glad to try, sir. You replied.",0.916970,0.891724,0.114075,0.847656,0.038544,neutral
2,3853-163249-0012,"Then they went back to their work, little drea...","Then they went back to their work, little drea...",0.917260,0.866662,0.001323,0.016998,0.981934,contradiction
3,3576-138058-0000,"Master and man dismounted from their beasts, a...","Master and man dismounted from their beasts, a...",0.944841,0.757205,0.008217,0.004005,0.987305,contradiction
4,3000-15664-0001,"Arctic beauty and desolation, with their bless...","Arctic beauty and desolation, with their bless...",0.995066,0.958548,0.051300,0.339355,0.609375,contradiction
5,2277-149897-0025,He was quite certain now that she knew he was ...,He was quite certain now that she knew he was ...,0.932387,0.650445,0.004467,0.911621,0.083984,neutral
6,3576-138058-0014,"Don Quixote was on foot, with his horse unbrid...",Don Quixote was on foot with his horse unbridl...,0.976126,0.787900,0.288818,0.256104,0.455566,contradiction


In [15]:
# Optional compact view of the main content-preservation metrics.
display(
    metrics_df[
        [
            "pair_id",
            "semantic_similarity",
            "TPI",
            "entailment",
            "neutral",
            "contradiction",
            "predicted",
        ]
    ].head(10)
)


,pair_id,semantic_similarity,TPI,entailment,neutral,contradiction,predicted
0,1462-170142-0001,0.853957,0.697676,0.205444,0.021011,0.773438,contradiction
1,251-137823-0004,0.982177,0.850371,0.998047,0.001343,0.000160,entailment
2,174-168635-0014,0.819615,0.486268,0.979004,0.012764,0.007988,entailment
3,3170-137482-0040,0.997856,1.000000,0.548828,0.422119,0.028793,entailment
4,7850-73752-0004,0.891451,0.809683,0.006142,0.986328,0.006985,neutral
5,1272-135031-0004,0.855182,0.433205,0.014870,0.075928,0.909180,contradiction
6,1993-147964-0000,0.971320,1.000000,0.982910,0.016037,0.001469,entailment
7,6241-61943-0000,0.660140,0.419391,0.004200,0.010719,0.984375,contradiction
8,3752-4943-0012,0.429771,0.000000,0.000220,0.000682,0.999023,contradiction
9,2035-147960-0015,0.541139,0.000000,0.003374,0.996094,0.000578,neutral
